In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("DSC100_Lab9.ipynb")

# Lab 09: Model Selection, Regularization, and Cross-Validation and Logistic Regression

This lab will consist of two parts: (1) Model selection, regularization, and cross-validation and (2) logistic regression. 

In the first part of the lab, you will practice using `scikit-learn` to generate models of various complexity. You'll then use the validation method and K-fold cross-validation to select the models that generalize the best.

In the second part of the lab, you will build logistic regression models to determine whether a tumor is benign or malignant based on some characteristics. 

To receive credit for a lab, answer all questions correctly and submit before the deadline.

## Collaboration Policy
Data science is a collaborative activity. While you may talk with others about this assignment, we ask that you **write your solutions individually**. If you discuss the assignment with others, please **include their names** in the cell below.

**Collaborators:** *list names here*

---
## ✅ Grading
Grading is broken down into auto-graded answers and free responses, which are not in this notebook. For auto-graded answers, the results of your code are compared to provided and/or hidden tests. For free response, readers will evaluate how well you answered the question (see written lab worksheet).

**Note that for ALL plotting questions, we will expect descriptive titles, axis labels, legends, etc. The following question serves as a good guideline on what is "enough": If I directly downloaded the plot and viewed it, would I be able to tell what was being visualized without knowing the question?** 

### Score breakdown



Question | Manual | Points
--- |---| ---
1a |No | 2
2a |No | 2
2b |Yes | 2
3a |No | 5
3b |Yes | 2
3c |Yes | 2
3d |No | 5
3e |Yes | 2
4 | No | 7
5 | No | 2
6a |No |5
6b| No| 2
6c| Yes | 2
7| No| 2
8a| No| 2
8b|No| 2
8c| Yes| 2
8b|No| 2

Total |18 |50

In [ ]:
# Run this cell to set up your notebook; no further action is needed.
import seaborn as sns
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
sns.set()
sns.set_context("talk")

import sklearn
import sklearn.datasets
import seaborn as sns
import plotly.offline as py
import plotly.graph_objs as go


from IPython.display import display, Latex, Markdown

<br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Introduction

For this lab, we will use a toy dataset to predict the house prices in Boston with data provided by the `sklearn.datasets` package. There are more interesting datasets in the package if you want to explore them during your free time!

Run the following cell to load the data. `pickle.load()` will return a dictionary object which includes keys for:
- `data` : The independent variables/features (X),
- `target` : The response vector (Y),
- `feature_names`: The column names,
- `DESCR` : A full description of the data, and
- `filename`: The name of the CSV file.

**Note**: The link to the dataset in the `DESC` no longer works.

In [ ]:
import pickle
boston_data = pickle.load(open("boston_data.pickle", "rb")) 

print("Dictionary keys:")
print(boston_data.keys())

print()

print("Sum of the attributes:")
print(sum(boston_data.data))

In [ ]:
print(boston_data['DESCR'])

A look at the `DESCR` attribute tells us the data contains these features:

    1. CRIM      per capita crime rate by town
    2. ZN        proportion of residential land zoned for lots over 
                 25,000 sq.ft.
    3. INDUS     proportion of non-retail business acres per town
    4. CHAS      Charles River dummy variable (= 1 if tract bounds 
                 river; 0 otherwise)
    5. NOX       nitric oxides concentration (parts per 10 million)
    6. RM        average number of rooms per dwelling
    7. AGE       proportion of owner-occupied units built prior to 1940
    8. DIS       weighted distances to five Boston employment centres
    9. RAD       index of accessibility to radial highways
    10. TAX      full-value property-tax rate per 10,000 USD
    11. PTRATIO  pupil-teacher ratio by town
    12. LSTAT    % lower status of the population
    
Let's now convert this data into a `pandas` `DataFrame`. 

In [ ]:
boston = pd.DataFrame(boston_data['data'], columns=boston_data['feature_names'])
boston.head()

<br>

---

### Question 1

Let's model this housing price data! Before we can do this, however, we need to split the data into training and validation sets. Remember that the response vector (housing prices) lives in the `target` attribute. A random seed is set here so we can deterministically generate the same splitting in the future if we want to test our result again and find potential bugs.

Use the sklearn's `train_test_split` [(documentation)](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function to split out 10% of the data for the validation set. Call the resulting splits `X_train`, `X_validation`, `Y_train`, and `Y_validation`.

In [ ]:
from sklearn.model_selection import train_test_split
np.random.seed(45) # DO NOT CHANGE

X = boston
Y = pd.Series(boston_data['target'])

X_train, X_validation, Y_train, Y_validation = ...

In [ ]:
grader.check("q1")

<br>

---

### Question 2a

As a warmup, fit a linear model to describe the relationship between the housing price and all available independent variables/features. We have imported `sklearn.linear_model` as `lm`, so you can use that instead of typing out the whole module name. Fill in the cell below to fit a model using the data contained in the training set. Then, use this fitted model to predict the housing prices for the validation set. We've written out the code to create a scatter plot showing the relationship between the predicted and actual housing prices in the validation set for you. 

In [ ]:
import sklearn.linear_model as lm

linear_model = lm.LinearRegression()

# Fit your linear model
# linear_model.fit(...)

# Predict housing prices on the validation set
Y_pred = ...

# DO NOT CHANGE THE CODE BELOW THIS LINE
# Plot true prices vs. predicted values
plt.scatter(Y_validation, Y_pred, alpha=0.5)

# Plot the x=y diagonal line
plt.plot([0, 50], [0, 50], color='r')
plt.xlabel("Prices $(y)$")
plt.ylabel("Predicted Prices $(\hat{y})$")
plt.title("Prices vs. Predicted Prices");

In [ ]:
grader.check("q2a")

<!-- BEGIN QUESTION -->

<br>

---

### Question 2b

Briefly analyze the scatter plot above. Do you notice any outliers? Write your answer in the cell below.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

Alternatively, we can plot the residuals against our model predictions (fitted values). This is known as the **residual plot**. Ideally, they would all be zero. Given the inevitability of noise, we would at least like them to be scattered randomly across the line where the residual is zero. By contrast, there appears to be a possible pattern with our model consistently underestimating prices for both very low and very high values, and possibly consistently overestimating prices towards the middle range below. We can figure out whether the model is overestimating or underestimating based on the sign of the residual: if it is negative, it means $y_i < \hat{y}_i$ so we are overestimating prices. Conversely, if the residual is positive, $y_i > \hat{y_i}$ and we end up underestimating prices.

In [ ]:
plt.scatter(Y_pred, Y_validation - Y_pred, alpha=0.5)
plt.ylabel("Residual $(y - \hat{y})$")
plt.xlabel("Predicted Prices $(\hat{y})$")
plt.title("Residuals vs. Predicted Prices")
plt.title("Residual of prediction for i-th house")
plt.axhline(y = 0, color='r');

<br>

---

### Question 3a

As we see in the scatter plots above, our model is not perfect. If it were perfect, the first scatter plot of predicted vs. true prices would follow the identity line (i.e., a line of slope 1 or $\hat{y} = y$). In contrast, the second scatter plot of residuals vs. predicted prices would be scattered randomly around the line where the residual is 0. To quantify the performance of the model, we will compute the Root Mean Squared Error (RMSE) of the predicted responses: 

$$
\textbf{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^n \left( y_i - \hat{y}_i \right)^2 }
$$

Fill in the function below to compute the RMSE. Then, assign `train_error` to the model’s RMSE when making predictions on the training data and `validation_error` to the model’s RMSE when making predictions on the validation set. Your implementation **should not** use `for` loops.

In [ ]:
def rmse(actual_y, predicted_y):
    """
    Args:
        predicted_y: An array of the predictions from the model.
        actual_y: An array of the ground truth labels.
        
    Returns:
        The root mean square error between the predictions and ground truth labels.
    """
    ...

train_error = ...
validation_error = ...

print("Training RMSE:", train_error)
print("Validation RMSE:", validation_error)

In [ ]:
grader.check("q3a")

<!-- BEGIN QUESTION -->

<br>

---

### Question 3b

Is your training error lower than the error on the validation set, which includes data the model never got to see while it was being trained? If so, why could this be happening? Answer in the cell below.


_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>
<hr style="border: 1px solid #fdb515;" />

## Overfitting

Sometimes we can get even higher accuracy by adding more features. For example, the code below adds the square, square root, and hyperbolic tangent of every feature to the design matrix. We've chosen these bizarre features specifically to highlight overfitting.

In [ ]:
boston_with_extra_features = boston.copy()
for feature_name in boston.columns:
    boston_with_extra_features[feature_name + "^2"] = boston_with_extra_features[feature_name] ** 2
    boston_with_extra_features["sqrt" + feature_name] = np.sqrt(boston_with_extra_features[feature_name])
    boston_with_extra_features["tanh" + feature_name] = np.tanh(boston_with_extra_features[feature_name])
    
boston_with_extra_features.head(5)

We split up our data again and refit the model. From this cell forward, we append `2` to the variable names `X_train, X_validation, Y_train, Y_validation, train_error, validation_error` in order to maintain our original data.

In [ ]:
np.random.seed(25)
X = boston_with_extra_features
X_train2, X_validation2, Y_train2, Y_validation2 = train_test_split(X, Y, test_size = 0.1)
linear_model.fit(X_train2, Y_train2);

Now, let's see the training and validation RMSE values:

In [ ]:
train_error2 = rmse(Y_train2, linear_model.predict(X_train2)) 
validation_error2 = rmse(Y_validation2, linear_model.predict(X_validation2))

print("Training RMSE:", train_error2)
print("Validation RMSE:", validation_error2)

Looking at our training and validation RMSE, we see that they are lower than you computed earlier. This strange model is seemingly better even though it includes seemingly useless features like the hyperbolic tangent of the average number of rooms per dwelling.

The code below generates the training and validation RMSE for 49 different models and stores the results in a `DataFrame`. The first model uses only the first feature "CRIM". The second model uses the first two features "CRIM" and "ZN", and so forth.

In [ ]:
errors_vs_N = pd.DataFrame(columns = ["N", "Training Error", "Validation Error"])

range_of_num_features = range(1, X_train2.shape[1] + 1)

# Iterates through different number of features
for N in range_of_num_features:
    # Use only the first N features for this model
    X_train_first_N_features = X_train2.iloc[:, :N]    
    linear_model.fit(X_train_first_N_features, Y_train2)
    train_error_overfit = rmse(Y_train2, linear_model.predict(X_train_first_N_features))
    
    # Preprocess validation set the same way as the training data
    X_validation_first_N_features = X_validation2.iloc[:, :N]
    validation_error_overfit = rmse(Y_validation2, linear_model.predict(X_validation_first_N_features))    
    
    # Save the RMSE
    errors_vs_N.loc[len(errors_vs_N)] = [N, train_error_overfit, validation_error_overfit]

errors_vs_N

If we plot the training and validation error as we add each additional feature, our training error gets lower and lower, and in fact, it's possible to prove with linear algebra that the training error will decrease monotonically.

By contrast, the error on unseen validation data is higher for the models with more parameters, since the lessons learned from these last 20+ features aren't actually useful when applied to unseen data. That is, these models aren't generalizable.

In [ ]:
import plotly.express as px
px.line(errors_vs_N, x = "N", y = ["Training Error", "Validation Error"], width=800, height=400)

<!-- BEGIN QUESTION -->

<br>

---

### Question 3c

This plot is a useful tool for **model selection**. Describe your observation in the plot above. Can this plot be used to inform the number of features you should use for modeling?

_Type your answer here, replacing this text._

<!-- END QUESTION -->


**Bonus:** You may be tempted to iterate through all possible feature combinations. While this may be possible when the dataset contains very few features, a dataset with 48 features has a total of ${48 \choose 1} + {48 \choose 2} + {48 \choose 3} + \cdots + {48 \choose 48}$ combinations. In practice, we often use heuristics like the example above that perform a reasonable amount of comparisons instead of trying out every combination due to computational constraints. 

<br>
<hr style="border: 1px solid #fdb515;" />

## Regularization

Regularization is the formal term that describes the process of limiting a model’s complexity, often with the aim of reducing overfitting and allowing for more generalizable models. Here, we consider Ridge Regression, also termed L2 Regularization (in contrast with LASSO Regression or L1 Regularization), as discussed in Lecture 16. Mathematically speaking, our objective function looks fairly similar to the usual formula for MSE with the addition of the regularization term: 

$$\min_{\theta} \frac{1}{n} || \mathbb{Y} - \mathbb{X}\theta || + \lambda \sum_{j=1}^{d} \theta_j^{2}$$

As an alternative and more realistic example, instead of using only the first N features, we can use various different regularization strengths. For example, for really low regularization strengths (e.g. $\lambda = 10^{-12}$), we get a model that is nearly identical to our linear regression model. This regularization term is so small you may even get a `LinAlgWarning`. Below, we import `Ridge`, which initializes a model similar to `lm.LinearRegression` but applies Ridge Regression.

**Note**: `sklearn` uses `alpha` to represent $\lambda$. The `alpha` parameter passed into `Ridge` represents the regularization parameter, and here, does not refer to the learning rate in the context of gradient descent.

In [ ]:
from sklearn.linear_model import Ridge

regularized_model = Ridge(alpha = 1e-12)
regularized_model.fit(X_train2, Y_train2)
ridge_coefs_low_regularization = regularized_model.coef_

linear_model = linear_model.fit(X_train2, Y_train2)
linear_coefs = linear_model.coef_

# Most of the differences are 0's
(ridge_coefs_low_regularization - linear_coefs).round()

However, if we pick a large regularization strength, e.g. $\lambda = 10^2$, we see that the resulting parameters are much smaller in magnitude. 

In [ ]:
regularized_model = Ridge(alpha = 10**2)
regularized_model.fit(X_train2, Y_train2)
regularized_model_coefs = regularized_model.coef_

# The difference in magnitude are mostly negative, 
# this shows that regularized_model has a smaller coefficient in magnitude
(abs(regularized_model_coefs) - abs(linear_coefs)).round()

### Standard Scaling / Normalization

Recall from the lecture that in order to properly regularize a model, the features should be at the same scale. Otherwise, the model has to spend more of its parameter budget to use "small" features (e.g., lengths in inches) compared to "large" features (e.g., lengths in kilometers).

To do this we can use sklearn's `StandardScaler` [(documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to create a new version of the `DataFrame` where every column has a mean of zero and a standard deviation of 1. 

**Optional:** Can you implement this using a `pandas` function?

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
ss.fit(boston_with_extra_features)
boston_with_extra_features_scaled = pd.DataFrame(ss.transform(boston_with_extra_features), \
                                                 columns = boston_with_extra_features.columns)
boston_with_extra_features_scaled

Let's now regenerate the training and validation sets using this new rescaled dataset.

In [ ]:
np.random.seed(25)
X = boston_with_extra_features_scaled
X_train3, X_validation3, Y_train3, Y_validation3 = train_test_split(X, Y, test_size = 0.10)

Fitting our regularized model with $\lambda = 10^2$ on this scaled data, we now see that our coefficients are of around the same magnitude. This is because all of our features are of around the same magnitude, whereas in the unscaled data, some of the features, like `TAX^2`, were much larger than others.

In [ ]:
regularized_model_standardized = Ridge(alpha = 10**2)
regularized_model_standardized.fit(X_train3, Y_train3)
regularized_model_standardized.coef_

<br>

---

### Question 3d: Finding an Optimum $\lambda$

In the cell below, write code that generates a `DataFrame` with the training and validation error for the range of lambdas given. Make sure you're using the 3rd training and validation sets, which have been rescaled, that is, `X_train3`, `X_validation3`, `Y_train3`, and `Y_validation3`. To lay out the process:

1. Initialize a new `lm.Ridge` model with the regularization hyperparameter set to the new value of $\lambda$.
2. Fit the model to the training set, `X_train3`.
3. Compute the RMSE of the model on the training and validation sets.
4. Add a row to the `DataFrame` containing the value of $\lambda$, training error, and validation error.

**Note**: You should use all 48 features for every single model that you fit, i.e. you're not going to be keeping only the first $N$ features.

**Hint**: It is possible to "append" or add a row to a `DataFrame` by calling `loc`, e.g. `df.loc[len(df)] = [2, 3, 4]` (assuming `df` has 3 columns!).

In [ ]:
error_vs_lambda = pd.DataFrame(columns = ["lambda", "Training Error", "Validation Error"])
range_of_lambda = 10**np.linspace(-5, 4, 40)

for lamb in range_of_lambda:
    regularized_model_lambda = ...
    ...
    train_error_overfit = ...
    holdout_error_overfit = ...
    error_vs_lambda.loc[len(error_vs_lambda)] = ...

error_vs_lambda.head()

In [ ]:
grader.check("q3d")

Below we plot your training and validation set error for the range of lambdas given. You should see a figure where training error decreases as model complexity increases, but the error on the validation set is large for extreme values of lambda and minimized for some intermediate values. Holding your mouse over the training error and validation error lines will display the lambda and value at that point.

Note that on your plot, the **x-axis is the inverse of complexity**! In other words, small lambda models (on the left) are complex, because there is no regularization. That's why the training error is lowest on the left side of the plot, as this is where overfitting occurs.

In [ ]:
px.line(error_vs_lambda, x = "lambda", y = ["Training Error", "Validation Error"], log_x=True, height = 400, width = 800)

<!-- BEGIN QUESTION -->

<br>

---

### Question 3e

Based on the plot above, which lambda value will you use in your model? Keep in mind that the x-axis in the plot above has a logarithmic scale.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>
<hr style="border: 1px solid #fdb515;" />

## REMINDER: Test Set vs. Validation Set (a.k.a. Development Set)

In the plots above, we trained our models on a training set and plotted the resulting RMSE on the training set in blue. We also held out a set of data and plotted the error on this validation set in red, calling it the "validation set error". 

For the example above, we used the validation set to pick a hyperparameter, so we'd also call this set a "development set". These terms are exactly synonymous.

It would not be accurate to call this line the "validation set error" because we did not use this dataset as a test set. While it is true that your code never supplied `X_validation3` or `Y_validation3` to the fit function of the ridge regression models, once you decide to use the validation set to select between different models, different hyperparameters, or different sets of features, then we are not using that dataset as a "test set" but rather a "validation set". That is, since we've used this set for picking lambda, the resulting errors are no longer unbiased predictors of our performance on unseen models -- the true error on an unseen dataset is likely to be somewhat higher than the validation set. After all, we trained 40 models and picked the best one!

In many real-world contexts, model builders will split their data into three sets: training, validation, and test sets, where the test set is *only* ever used once. That is, there are two holdout sets: one used as a development set (for model selection), and one used as a test set (for providing an unbiased estimate of error).

## An Alternate Strategy for Hyper Parameter Selection: K-Fold Cross Validation

Earlier, we used the holdout method for model selection (the holdout method is also sometimes called "simple cross-validation"). Another approach is K-fold cross-validation. This allows us to use more data for training instead of having to set aside some specifically for hyperparameter selection. However, doing so requires more computation resources, as we'll have to fit K models per hyperparameter choice.

In our course, there's really no reason not to use cross-validation. However, in environments where models are very expensive to train (e.g. deep learning), you'll typically prefer using a holdout set (simple cross-validation) rather than K-fold cross-validation.

To emphasize what K-fold cross-validation actually means, we're going to manually carry out the procedure. Recall the approach looks something like the figure below for 4-fold cross-validation:

<img src="cv.png" width=800px alt="4-fold cross validation">

When we use K-fold cross-validation, rather than using a validation set for model selection, we instead use the training set for model selection. To select between various features, models, or hyperparameters, we split the training set further into multiple temporary train and validation sets (each split is called a "fold", hence K-fold cross-validation). We will use the average validation error across all K folds to make our optimal feature, model, and hyperparameter choices. In this example, we'll only use this procedure for hyperparameter selection, specifically to choose the best lambda.

<br>

---

### Question 4

Scikit-learn has built-in support for cross-validation. However, to better understand how cross-validation works, complete the following function which cross-validates a given model.

1. Use `sklearn`'s `KFold.split` ([documentation](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)) function to get 4 splits on the training data. Note that `split` returns the indices of the data for that split.
2. For **each** split:
    1. Select the training and validation rows and columns based on the split indices and features.
    2. Compute the RMSE on the validation split.
    3. Return the average error across all cross-validation splits.


In [ ]:
from sklearn.model_selection import KFold

def compute_CV_error(model, X_train, Y_train):
    '''
    Split the training data into 4 subsets.
    For each subset, 
        - Fit a model holding out that subset.
        - Compute the MSE on that subset (the validation set).
    You should be fitting 4 models in total.
    Return the average MSE of these 4 folds.

    Args:
        model: An sklearn model with fit and predict functions. 
        X_train (DataFrame): Training data.
        Y_train (DataFrame): Label.
    
    Return:
        The average validation MSE for the 4 splits.
    '''
    kf = KFold(n_splits=4)
    validation_errors = []
    
    for train_idx, valid_idx in kf.split(X_train):
        # split the data
        split_X_train, split_X_valid = ..., ...
        split_Y_train, split_Y_valid = ..., ...

        # Fit the model on the training split
        ...
        
        # Compute the RMSE on the validation split
        error = ...


        validation_errors.append(error)
        
    return np.mean(validation_errors)

In [ ]:
grader.check("q4")

<br>

---

### Question 5

Use `compute_CV_error` to add a new column to `error_vs_lambda` which gives the 4-fold cross-validation error for the given choices of $\lambda$ in `range_of_lambda`. `cv_errors` should be a list or an array of cross-validation errors generated by `compute_CV_error` for each tested value of $\lambda$. Again, use the 3rd training and validation sets `X_train3`, `X_validation3`, `Y_train3`, and `Y_validation3`.

In [ ]:
cv_errors = []

...

error_vs_lambda["CV Error"] = cv_errors
error_vs_lambda.head()

In [ ]:
grader.check("q5")

The code below shows the holdout error that we computed in the previous problem as well as the 4-fold cross-validation error. Note that the cross-validation error shows a similar dependency on lambda relative to the holdout error. This is because they are both doing the same thing: trying to estimate the expected error on unseen data drawn from the distribution from which the training set was drawn. 

In other words, this figure compares the holdout method with 4-fold cross-validation.

In [ ]:
px.line(error_vs_lambda, x = "lambda", y = ["Validation Error", "CV Error"], 
        log_x=True, height = 400, width = 800)

<br/><br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 2: Logistic Regression



In this lab, we will manually construct the logistic regression model and minimize cross-entropy loss using `scipy.minimize`. This structure mirrors the linear regression labs from earlier in the semester and lets us dive deep into how logistic regression works. We also introduce the `sklearn.linear_model.LogisticRegression` module that you would use in practice, and we explore performance metrics for classification.


## Data Loading

We will explore a breast cancer dataset from the University of Wisconsin ([source](https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(diagnostic))). This dataset can be loaded using the `sklearn.datasets.load_breast_cancer()` method.  

In [ ]:
# Run this cell to load the data; no further action is needed.
data = sklearn.datasets.load_breast_cancer()

# Data is a dictionary.
print(data.keys())
print(data.DESCR)

<br/>

Since the data format is a dictionary, we will perform some preprocessing to create a `DataFrame`.

In [ ]:
# Run this cell to see the first five rows of the data; no further action is needed.
df = pd.DataFrame(data.data, columns=data.feature_names)
df.head()

The prediction task for this data is to predict whether a tumor is benign or malignant (a binary decision), given the characteristics of that tumor. The prediction labels are stored in the field `data.target`. To put the data back in its original context, we will create a new column called `"malignant"` which will be 1 if the tumor is malignant and 0 if it is benign (reversing the definition of `target`).

In this lab, we will fit a simple **classification model** to predict breast cancer from the cell nuclei of a breast mass. For simplicity, we will work with only one feature: the `mean radius` which corresponds to the size of the tumor. Our output (i.e., response) is the `malignant` column.

In [ ]:
# Run this cell to define X and Y; no further action is needed.

# Target data_dict['target'] = 0 is malignant 1 is benign
df['malignant'] = (data.target == 0).astype(int)

# Define our features/design matrix X
X = df[["mean radius"]]
Y = df['malignant']

<br/>

Before we go further, we will split our dataset into training and testing sets. This lets us explore the prediction power of our trained classifier on both seen and unseen data.

In [ ]:
# Run this cell to create a 75-25 train-test split; no further action is needed. 
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.25, random_state=42)
print(f"Training Data Size: {len(X_train)}")
print(f"Test Data Size: {len(X_test)}")

<br/>
<hr style="border: 1px solid #fdb515;" />

## Part 1: Defining the Model

In these first two parts, you will manually build a logistic regression classifier.

Recall that the Logistic Regression model is written as follows:

$$p = \Large f_{\theta}(x) = \sigma ( x^T \theta)$$

where $f_{\theta}(x)= P(Y=1 | x)$ is the probability that our observation belongs to class 1, and $\sigma$ is the sigmoid activation function:

$$\Large \sigma(z) = \frac{1}{1 + e^{-z}}$$

If we have a single feature, then $x$ is a scalar and our model has parameters $\theta^T = [\theta_0 \ \ \theta_1]$ as follows:

$$\Large f_{\theta}(x) = \sigma (\theta_0 + \theta_1 x)$$

Therefore just like OLS, if we have $n$ data points and $d$ features, then we can construct the design matrix
$$\Large \mathbb{X} \in \mathbb{R}^{n \times (d + 1)}$$
with an all-ones column. Run the below cell to construct `X_intercept_train`. The syntax should look familiar:

In [ ]:
# Run this cell to add the bias column; no further action is needed.
def add_bias_column(X):
    return np.hstack([np.ones((len(X), 1)), X])

X_intercept_train = add_bias_column(X_train)
X_intercept_train.shape

<br/>

---

### Question 6a

Using the above definition for $\mathbb{X}$, we can also construct a matrix representation of our Logistic Regression model, just like we did for OLS. Noting that $\theta^T = [\theta_0 \ \ \theta_1\ \ \dots \ \ \theta_d$], the vector $\hat{\mathbb{Y}}$ is:

$$\Large \hat{\mathbb{Y}} = \sigma(\mathbb{X} \theta) $$

Then the $i$-th element of $\hat{\mathbb{Y}}$ is the probability that the $i$-th observation belongs to class 1, given the feature vector is the $i$-th row of design matrix $\mathbb{X}$, and the parameter vector $\theta$.

Below, implement the `lr_model` function to evaluate this expression. To matrix-multiply two `numpy` arrays, use `@` or `np.dot`. In case you're interested, the [matmul documentation](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) contrasts the two methods.



In [ ]:
def sigmoid(z):
    """
    The sigmoid function is defined for you.
    """
    return 1 / (1 + np.exp(-z))

def lr_model(theta, X):
    """
    Returns the logistic regression model as defined above.
    You should not need to use a for loop; use @ or np.dot.
    
    Args:
        theta: The model parameters. Dimension (d+1,).
        X: The design matrix. Dimension (n, d+1).
    
    Return:
        Probabilities that Y = 1 for each data point.
        Dimension (n,).
    """
    ...

In [ ]:
grader.check("q6a")

<br/>

---

###  Question 6b: Compute Empirical Risk
Now let's try to analyze the cross-entropy loss from logistic regression. Suppose for a single observation, we predict probability $p$ that the true response $y$ is in class 1 (otherwise the prediction is 0 with probability $1 - p$). The cross-entropy loss is $-log(p)$ when $y=1$ and $-log(1-p)$ when $y=0$. More concretely:

$$ \text{CE Loss} = - \left( y \log(p) + (1 - y) \log(1 - p) \right)$$

For the logistic regression model, the **empirical risk** is therefore defined as the average cross-entropy loss across all $n$ data points:

$$R(\theta) = -\frac{1}{n} \sum_{i=1}^n \left( y_i \log(\sigma(X_i^T \theta)) + (1 - y_i) \log(1 - \sigma(X_i^T \theta))  \right) $$

Where $y_i$ is the $i-$th response in our dataset, $\theta$ are the parameters of our model, $X_i^T$ is the $i$-th row of our design matrix $\mathbb{X}$, and $\sigma(X_i^T \theta)$ is the probability that the response is 1 given input $X_i$.

Below, implement the function `lr_loss` that computes empirical risk over the dataset. Feel free to use the functions defined in the previous part.

In [ ]:
def lr_avg_loss(theta, X, Y):
    '''
    Compute the average cross-entropy loss using X, Y, and theta.
    You should not need to use a for loop. 

    Args:
        theta: The model parameters. Dimension (d+1,).
        X: The design matrix. Dimension (n, d+1).
        Y: The label. Dimension (n,).

    Return:
        The average cross-entropy loss.
    '''
    ...
    ...

In [ ]:
grader.check("6b")

<br/>

Below is an interactive plot showing the average training cross-entropy loss for various values of $\theta_0$ and $\theta_1$ (respectively x and y axis in the plot). You may receive a `Javascript Error: Something went wrong with axis scaling` error. If your image does not show up, there are two potential workarounds: (1) run the following cell below to generate a static version of the plot and check out the interactive plot in the walkthrough video, or (2) restart your kernel (upper left menu -> `Kernel` -> `Restart Kernel and Run up to Selected Cell...`). 

In [ ]:
# Run this cell to create the plotly visualization. 
# If this gives a  Javascript Error, run the cell below instead. 
with np.errstate(invalid='ignore', divide='ignore'):
    uvalues = np.linspace(-8,8,70)
    vvalues = np.linspace(-5,5,70)
    (u,v) = np.meshgrid(uvalues, vvalues)
    thetas = np.vstack((u.flatten(),v.flatten()))
    lr_avg_loss_values = np.array([lr_avg_loss(t, X_intercept_train, Y_train) for t in thetas.T])
    lr_loss_surface = go.Surface(name="Logistic Regression Loss",
            x=u, y=v, z=np.reshape(lr_avg_loss_values,(len(uvalues), len(vvalues))),
            contours=dict(z=dict(show=True, color="gray", project=dict(z=True)))
        )
    fig = go.Figure(data=[lr_loss_surface])
    fig.update_layout(
        scene = dict(
            xaxis_title='theta_0',
            yaxis_title='theta_1',
            zaxis_title='Loss'),
            width=700,
            margin=dict(r=20, l=10, b=10, t=10))
    py.iplot(fig)

In [ ]:
# Run this cell to create the plotly visualization; no further action is required. 
from matplotlib import cm

with np.errstate(invalid='ignore', divide='ignore'):
    fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(10,10))
    
    uvalues = np.linspace(-8,8,70)
    vvalues = np.linspace(-5,5,70)
    u,v = np.meshgrid(uvalues, vvalues)
    thetas = np.vstack((u.flatten(),v.flatten()))
    lr_avg_loss_values = np.array([lr_avg_loss(t, X_intercept_train, Y_train) for t in thetas.T])


    # Plot the surface.
    surf = ax.plot_surface(u, v, np.reshape(lr_avg_loss_values,(len(uvalues), len(vvalues))), 
                           cmap=cm.coolwarm, linewidth=0, antialiased=False)

    # Set the azimuth and elevation angles
    ax.view_init(azim=35, elev=30)
    
    # customize 
    plt.xlabel('theta_0', fontsize=15, labelpad=15)
    plt.ylabel('theta_1', fontsize=15, labelpad=15)
    plt.title('Loss')
    
    # Add a color bar which maps values to colors.
    fig.colorbar(surf, shrink=0.5, aspect=5)
    plt.show()
    

<!-- BEGIN QUESTION -->

<br/>

---

### Question 6c
Describe one interesting observation about the loss plot above.


_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>
<hr style="border: 1px solid #fdb515;" />

## Part 2: Fit and Predict

### [Tutorial] `scipy.optimize.minimize`

The next two cells call the `minimize` function from `scipy` on the `lr_avg_loss` function you defined in the previous part. We pass in the training data to `args` ([documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)) to find the `theta_hat` that minimizes the average cross-entropy loss over the training set.

In [ ]:
# Run this cell to minimize lr_avg_loss using scipy; no further action is needed.
from scipy.optimize import minimize

min_result = minimize(lr_avg_loss,
                      x0=np.zeros(X_intercept_train.shape[1]),
                      args=(X_intercept_train, Y_train))
min_result

In [ ]:
# Run this cell to print `theta_hat`; no further action is needed.
theta_hat = min_result['x']
theta_hat

<br/>

Because our design matrix $\mathbb{X}$ leads with a column of all ones, `theta_hat` has two elements: $\hat{\theta}_0$ is the estimate of the intercept/bias term, and $\hat{\theta}_1$ is the estimate of the slope of our single feature.

<br/>

### Recap:
* For logistic regression with the parameter vector $\theta$, $P(Y = 1 | x) = \sigma(x^T \theta)$, where $\sigma$ is the sigmoid function and $x$ is a feature vector. Therefore $\sigma(x^T \theta)$ is the probability that the response is 1 given the feature is $x$. Otherwise the response is 0 with probability $P(Y = 0 | x) = 1 - \sigma(x^T \theta)$.
* The $\hat{\theta}$ *minimizes* average cross-entropy loss of our training data.

The main takeaway is that logistic regression models **probabilities** of classifying data points as 1 or 0. Next, we use this takeaway to implement model predictions.

<br/>

---

## Question 7

Using the `theta_hat` estimate above, we can construct a **decision rule** for classifying a data point with observation $x$. Let $P(Y = 1 | x) = \sigma(x^T \hat{\theta})$:

$$ \text{classify}(x) = \begin{cases} 
1, &\quad\text{if}\ \  P(Y = 1 | x) \geq 0.5 \\ 
0, &\quad\text{if}\ \ P(Y = 1 | x) < 0.5 
\end{cases}$$

This decision rule has a decision **threshold** $T = 0.5$. This threshold means that we treat the classes $0$ and $1$ "equally." Lower thresholds mean that we are more likely to predict $1$, whereas higher thresholds mean that we are more likely to predict $0$.

<br/>

Implement the `lr_predict` function below, which returns a vector of predictions according to the logistic regression model. The function  takes a design matrix of observations `X`, parameter estimate `theta`, and decision threshold `threshold` with a default value of 0.5.

In [ ]:
def lr_predict(theta, X, threshold=0.5):
    '''
    Classification using a logistic regression model
    with a given decision rule threshold.

    Args:
        theta: The model parameters. Dimension (d+1,)
        X: The design matrix. Dimension (n, d+1).
        threshold: Decision rule threshold for predicting class 1.

    Return:
        A vector of predictions.
    '''
    ...

# Do not modify below this line.
Y_train_pred = lr_predict(theta_hat, X_intercept_train)
Y_train_pred

In [ ]:
grader.check("q7")

<br/>

### [Tutorial] Linearly separable data

How do these predicted classifications compare to the true responses $\mathbb{Y}$?

Run the cell below to visualize our predicted responses, the true responses, and the probabilities we used to make predictions. We use `sns.stripplot` which introduces some jitter to avoid overplotting.

In [ ]:
# Run this cell to generate the visualization; no further action is needed.
plot_df = pd.DataFrame({"X": np.squeeze(X_train),
                        "Y": Y_train,
                        "Y_pred": Y_train_pred,
                        "correct": (Y_train == Y_train_pred)})
sns.stripplot(data=plot_df, x="X", y="Y", orient='h', alpha=0.8, hue="correct")
plt.xlabel('mean radius, $x$')
plt.ylabel('$y$')
plt.yticks(ticks=[0, 1], labels=['0:\nbenign', '1:\nmalignant'])
plt.title("Predictions for decision threshold T = 0.5")
plt.show()

<br/>

Because we are using a decision threshold $T = 0.5$, we predict $1$ for all $x$ where our predicted probability $\sigma(x^T\theta)$ is greater than or equal to 0.5. Writing this out mathematically and solving for $x^T\theta$: 
\begin{align}
\frac{1}{1+e^{-x^T\theta}} &= \frac{1}{2} \\ 
2 &= 1 + e^{-x^T\theta} \\
1 &= e^{-x^T\theta} \\
log(1)  &= log(e^{-x^T\theta})  \\
0 &= -x^T\theta  \\
\end{align}

We see that our decision threshold is when $x^T\theta = 0$. For the single mean radius feature, we can use algebra to solve for the boundary to be approximately $x \approx 14.8$. We can see this by substituting for $\theta = \hat{\theta}$ in the equation above:
$$x^T\hat{\theta} = 0 $$
$$
\begin{matrix}\begin{bmatrix}1 & x\end{bmatrix}\\\mbox{}\end{matrix}\begin{bmatrix} \hat{\theta_0} \\ \hat{\theta_1} \end{bmatrix} = 0\\
$$

From the minimize function, we found that `theta_hat` is `array([-13.87178638,   0.93723916])`. Plugging for $\hat{\theta}$:

$$-13.87178638 +  0.93723916x = 0$$ 

$$x \approx 14.8$$


In other words, we will always predict $0$ (benign) if the mean radius feature is less than 14.8 and $1$ (malignant) otherwise. However, in our training data, there are data points with large mean radii that are benign and vice versa. Our data is not **linearly separable** by a vertical line.

The above visualization is useful when we have just one feature. In practice, however, we use other performance metrics to diagnose our model performance. Next, we will explore several metrics: accuracy, precision, recall, and confusion matrices.

<br/>
<hr style="border: 1px solid #fdb515;" />

## Part 3: Quantifying Performance



### [Tutorial] sklearn's `LogisticRegression`
Instead of using the model structure that we built manually in the previous questions, we will instead use `sklearn`'s `LogisticRegression` function, which operates similarly to the `sklearn` OLS, Ridge, and LASSO models.

Let's first fit a logistic regression model to the training data. Some notes: 
* Like with linear models, the `fit_intercept` argument specifies if the model includes an intercept term. We therefore pass in the original matrix `X_train` (defined at the beginning of the notebook, without intercept term) in the call to `lr.fit()`.
* `sklearn` fits an **L2 regularized** logistic regression model by default as specified in the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for more details. The `penalty` argument specifies the regularization penalty term. 

In [ ]:
# Run this cell to fit a sklearn LogisticRegression model; no further action is needed. 
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
        fit_intercept=True,
        penalty='l2')

lr.fit(X_train, Y_train)
lr.intercept_, lr.coef_

<br/>

Note that because we are now fitting a regularized logistic regression model, the estimated coefficients above deviate slightly from our numerical findings in Question 1.

<br/>

Like with linear models, we can call `lr.predict(x_train)` to classify our training data with our fitted model.

In [ ]:
# Run this cell to make predictions; no further action is needed. 
lr.predict(X_train)

Note that for a binary classification task, the `sklearn` model uses an unadjustable decision rule of 0.5. If you're interested in manually adjusting this threshold, check out the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba) for `lr.predict_proba()`.

<br/>

---

### Question 8a: Accuracy

Fill in the code below to compute the training and testing accuracy, defined as:

$$
\text{Training Accuracy} = \frac{1}{n_{train\_set}} \sum_{i \in {train\_set}} {\mathbb{1}_{y_i = \hat{y_i}}}
$$

$$
\text{Testing Accuracy} = \frac{1}{n_{test\_set}} \sum_{i \in {test\_set}} {\mathbb{1}_{y_i = \hat{y_i}}}
$$

where for the $i$-th observation in the respective dataset, $\hat{y_i}$ is the predicted response (class 0 or 1), and $y_i$ is the true response.  $\mathbb{1}_{y_i = \hat{y_i}}$ is an indicator function which is $1$ if ${y_i} = \hat{y_i}$ and $ 0$ otherwise.

In [ ]:
train_accuracy = ...
test_accuracy = ...

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

In [ ]:
grader.check("q8a")

<br/>

---

### Question 8b: Precision and Recall

It seems we can get a very high test accuracy. What about precision and recall?  
- **Precision** (also called positive predictive value) is the fraction of true positives among the total number of data points predicted as positive.  
- **Recall** (also known as sensitivity) is the fraction of true positives among the total number of data points with positive labels.

Precision measures the ability of our classifier to avoid predicting negative samples as positive (i.e., avoid false positives), while recall is the ability of the classifier to find all the positive samples (i.e., avoid false negatives).

Below is a graphical illustration of precision and recall, modified slightly from [Wikipedia](https://en.wikipedia.org/wiki/Precision_and_recall):

<img src="precision_recall.png" alt="precision_recall" width="600">

Mathematically, Precision and Recall are defined as:
$$
\text{Precision} = \frac{n_{true\_positives}}{n_{true\_positives} + n_{false\_positives}} = \frac{TP}{TP+FP}
$$

$$
\text{Recall} = \frac{n_{true\_positives}}{n_{true\_positives} + n_{false\_negatives}}=\frac{TP}{TP+FN}
$$

Use the formulas above to compute the precision and recall for the **test set** using the `lr` model trained using `sklearn`.

In [ ]:
Y_test_pred = ...

precision = ...
recall = ...

print(f'precision = {precision:.4f}')
print(f'recall = {recall:.4f}')

In [ ]:
grader.check("q8b")

<br/>

Our precision is fairly high, while our recall is a bit lower.

Consider the following plots, which display the distribution of the **response variable** $\mathbb{Y}$ in the training and test sets. Recall class labels are 0: benign, 1: malignant.

In [ ]:
fig, axes = plt.subplots(1, 2)
sns.countplot(x=Y_train, ax=axes[0]);
sns.countplot(x=Y_test, ax=axes[1]);

axes[0].set_title('Train')
axes[1].set_title('Test')
plt.tight_layout();

<!-- BEGIN QUESTION -->

<br>

---

### Question 8c
Based on the above distribution, what might explain the observed difference between our precision and recall metrics?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>


### [Tutorial] Confusion Matrices

To understand the link between precision and recall, it's useful to create a **confusion matrix** of our predictions. Luckily, `sklearn.metrics` provides us with such a function!

The `confusion_matrix` function ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)) categorizes counts of data points based if their true and predicted values match.

For the 143-datapoint test dataset:

In [ ]:
# Run this cell to define the confusion matrix; no further action is needed.
from sklearn.metrics import confusion_matrix

Y_test_pred = lr.predict(X_test)
cnf_matrix = confusion_matrix(Y_test, Y_test_pred)
cnf_matrix

We've implemented the following function to better visualize these four counts against the true and predicted categories:

In [ ]:
# Run this cell to plot the confusion matrix; no further action is needed.

def plot_confusion_matrix(cm, classes,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    """
    import itertools

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    plt.grid(False)

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, np.round(cm[i, j], 2),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    
class_names = ['False', 'True']

plot_confusion_matrix(cnf_matrix, classes=class_names,
                      title='Confusion matrix, without normalization')

<br>

---

### Question 8d: Normalized Confusion Matrix

To better interpret these counts, assign `cnf_matrix_norm` to a **normalized confusion matrix** by the count of each true label category.

In other words, build a 2-D `numpy` array constructed by normalizing `cnf_matrix` by the count of data points in each row. For example, the top-left quadrant of `cnf_matrix_norm` should represent the proportion of true negatives over the total number of data points with negative labels. 

**Hints**: 
* When adding values in a 2-D array `arr`, `arr.sum(axis=0)` will calculate the sum of the columns while `arr.sum(axis=1)` will calculate the sum of the rows. 
* In array broadcasting, you may encounter issues dividing 2-D `numpy` arrays by 1-D `numpy` arrays. 
    * Check out the `keepdims` parameter in `np.sum` ([documentation](https://numpy.org/doc/stable/reference/generated/numpy.sum.html)), to preserve the dimensions of `cnf_matrix` after using `np.sum` on it.
    * Alternatively, add the dimension back using `np.newaxis` ([documentation](https://numpy.org/doc/stable/reference/constants.html#numpy.newaxis)).

In [ ]:
cnf_matrix_norm = ...

# Do not modify below this line.
plot_confusion_matrix(cnf_matrix_norm, classes=class_names,
                       title='Normalized confusion matrix')

In [ ]:
grader.check("q8d")

<br/>

Compare the normalized confusion matrix to the values you computed for precision and recall earlier:

In [ ]:
# Run this cell to see precision and recall again; no further action is needed.
print(f'precision = {precision:.4f}')
print(f'recall = {recall:.4f}')

<br/>
Based on the definitions of precision and recall, why does only recall appear in the normalized confusion matrix? Why doesn't precision appear? (No answer required for this part; just something to think about.)

### Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Submit this file to the Lab 09 assignment on Gradescope.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)